# Expression analysis in TCGA DATA with AnnData

This notebook will demonstrate how to analyze TCGA data in the AnnData form, with DeSeq analysis and some visualization.

In [ ]:
import anndata as ad    # For dealing with AnnData file formats
import holoviews as hv  # For HoloViews plotting
import hvplot.pandas    # noqa
import numpy as np      # Calculations around the AnnData objects
import pooch            # For file downloading and convenient caching
import pydeseq2.dds     # DeSeq datasets
import pydeseq2.ds      # DeSeq stats

import hv_anndata
from hv_anndata import ClusterMap

hv_anndata.register()
hv.extension("bokeh")

## Load data

We will be using TCGA breast cancer (BRCA) samples, which have been randomly sampled to approximately 120MB in size for easier demonstration.

The primary matrix for the AnnData object, `X`, consists of read counts from the CRDC for these samples.

In [ ]:
brca_test_file_path = pooch.retrieve(
    url="https://storage.googleapis.com/tcga-anndata-public/test2025-04/brca_test.h5ad",
    known_hash="md5:0e17ecf3716174153bc31988ba6dd161"
)
exp = ad.read_h5ad(brca_test_file_path)

# Select a cohort

First, we will examine the AnnData to see which types of metadata are associated with samples, which appear in the `obs` matrix. Information about the genes is in the `var` matrix. Additional information is sometimes held in the `obsm` key of the AnnData object, and can contain multiple different associated DataFrames.

In [ ]:
exp

The `sample_type_name` column seems promising for describing the samples and filtering for the appropriate type.

In [ ]:
exp.obs.sample_type_name.value_counts()

To simplify this analysis, we'll discard the metastatic samples, since there are only two.

We can compare the gene expression in primary tumor to normal samples, for a basic analysis of genes that are differentiall expressed in breast cancer.

In [ ]:
sample_types = ["Primary Tumor", "Solid Tissue Normal"]

The `primary_site` factor may be of interest as well, let's see what is in there:

In [ ]:
exp.obs.primary_site.value_counts()

Now, let's use AnnData indexing to select just the samples of interest.

Then, we'll restrict the genes to only those with a mean value greater than 50 read counts for further analysis. The `X` matrix for the dataset uses read counts from the original TCGA breast cancer dataset.

Finally, we'll convert `X` from a sparse matrix to a dense matrix to prepare for differential expression analysis.

In [ ]:
brca = exp[(exp.obs.primary_site == "Breast") & (exp.obs.sample_type_name.isin(sample_types))]
brca = brca[:, np.mean(brca.X, axis=0) > 50].copy()
brca.X = brca.X.todense()

Now that we have pre-filtered our AnnData to the samples and genes of highest relevance, we'll use PyDeSeq2 to find genes that are differntially expressed in breast cancer. The `design` attribute is a formula describing which of the `obs` variables to use to group the variables. By specifying `sample_type_name` we will compare the `Primary Tumor` group to the `Solid Tissue Normal` group.

This function is somewhat chatty, as very large datasets can run for quite some time.

In [ ]:
brca_ds = pydeseq2.dds.DeseqDataSet(adata = brca, design="~sample_type_name")
brca_ds.deseq2()

Now we have a PyDeSeq2 DataSet object. Let's add log transformation to the normalized courts so that visualization and comparisons happen on a more useful scale. The `log1p` function adds 1 to each count before applying the logarithm, which allows the frequent count value of 0 to remain a finite value instead of negative infinity.

In [ ]:
brca_ds.layers['log1p'] = np.log1p(brca_ds.layers['normed_counts'])

Next, lets calculate statistics for each gene's differential expression.

In [ ]:
t_n = pydeseq2.ds.DeseqStats(brca_ds, contrast=["sample_type_name"] + sample_types)
t_n.summary()

Now let's copy the results per gene into a more convenient data frame, convert the adjusted pvalue to a more convenient scale for plotting, and compare the statistical significance on the y-axis to the effect size on the x-axis, as measured bythe log2 fold change.

In [ ]:
t_n_res = t_n.results_df
t_n_res

t_n_res['neg_log10_p'] = -np.log10(t_n_res['pvalue'])
t_n_res['neg_log10_padj'] = -np.log10(t_n_res['padj'])
volcano_plot = t_n_res.hvplot.scatter(x="log2FoldChange", y="neg_log10_padj")
volcano_plot

Let's add some annotation on the plot for the significance level of padj == 0.05, and a log2 fold change of 1.

In [ ]:
(
    volcano_plot
    * hv.HLine(-np.log10(0.05)).opts(color='red', line_dash='dashed')
    * hv.VLine(-1).opts(color='blue', line_dash='dashed')
    * hv.VLine(1).opts(color='blue', line_dash='dashed')
)

Now we can select the statistically significant genes with large effect size:

In [ ]:
sig_genes = t_n_res[(t_n_res['neg_log10_padj'] > - np.log10(0.05)) & (abs(t_n_res['log2FoldChange']) > 1.0)]
sig_genes

And lets visualize the heatmap of these genes with dendrograms. First, we will log1p transform the counts matrix, to make the scale more amenable to visualization.

In [ ]:
brca_log1p = brca
brca_log1p.X = np.log1p(brca_log1p.X)
ClusterMap(adata=brca_log1p[:, sig_genes.index])